In [ ]:
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq xvfb fluxbox x11vnc novnc websockify > /dev/null 2>&1

# Install Google Chrome (not pre-installed in Colab)
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq /tmp/chrome.deb > /dev/null 2>&1

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Confirm installs
import shutil
for tool in ["Xvfb", "x11vnc", "fluxbox", "websockify", "google-chrome", "cloudflared"]:
    found = shutil.which(tool)
    status = "✅" if found else "❌"
    print(f"{status} {tool}: {found or 'NOT FOUND'}")

In [ ]:
import os, time, subprocess

# Kill any leftover processes
os.system("pkill -9 -f 'Xvfb|x11vnc|fluxbox|websockify|chrome|cloudflared' 2>/dev/null || true")
time.sleep(2)

# 1. Start virtual display
subprocess.Popen(
    ["Xvfb", ":1", "-screen", "0", "1280x800x24", "+extension", "GLX", "+render", "-noreset"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ Xvfb started")

# 2. Start window manager
subprocess.Popen(
    ["fluxbox", "-display", ":1"],
    env={**os.environ, "DISPLAY": ":1"},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ Fluxbox started")

# 3. Start VNC server (no password, port 5900)
subprocess.Popen(
    ["x11vnc", "-display", ":1", "-forever", "-nopw", "-shared", "-rfbport", "5900"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ x11vnc started")

# 4. Find noVNC web files
import shutil, pathlib
novnc_candidates = ["/usr/share/novnc", "/usr/local/share/novnc"]
novnc_path = next((p for p in novnc_candidates if pathlib.Path(p).exists()), None)

if not novnc_path:
    raise RuntimeError("noVNC web directory not found! Re-run Cell 1.")
print(f"✅ noVNC path: {novnc_path}")

# 5. Start noVNC WebSocket bridge on port 6080
subprocess.Popen(
    ["websockify", "--web", novnc_path, "6080", "localhost:5900"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ websockify started on port 6080")

# 6. Launch Chrome on the virtual display
subprocess.Popen(
    [
        "google-chrome",
        "--no-sandbox",
        "--disable-setuid-sandbox",
        "--disable-dev-shm-usage",
        "--start-maximized",
        "--display=:1",
        "https://www.google.com"
    ],
    env={**os.environ, "DISPLAY": ":1"},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)
print("✅ Chrome launched")

print("\n🟢 All services running. Now run Cell 3 to get your public URL.")

In [ ]:
import subprocess, re, threading, time, os

os.system("pkill -9 cloudflared 2>/dev/null || true")
time.sleep(1)

def start_tunnel():
    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:6080"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in process.stdout:
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🌐 OPEN THIS URL IN YOUR LAPTOP BROWSER:")
            print(f"\n   {url}/vnc.html?autoconnect=true\n")
            print("="*60)
            print("\nIf the screen looks black, wait 5 seconds and refresh.")
            print("Chrome should appear inside the browser window.")

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()
t.join()